# Aria Custom Domain Extensions & Plugins Guide

**Guide for building domain-specific AI applications, custom providers, and plugin extensions.**

Learn how to extend Aria for specialized domains like healthcare, finance, education, and more.


## Domain-Specific AI Architecture

### Plugin System Overview

```
Core Aria Platform
├─ Base Providers (OpenAI, Azure, LMStudio)
├─ Shared Infrastructure (DB, Cache, Memory)
└─ REST API Layer
    ↓
Plugin Interface
├─ Custom Provider Plugin
├─ Custom Retriever Plugin (RAG)
├─ Custom Character Behavior Plugin
├─ Domain-Specific Tool Plugin
└─ Custom Inference Engine
    ↓
Domain Implementations
├─ Healthcare Domain
├─ Finance Domain
├─ Education Domain
└─ Custom Industry-Specific
```

### Healthcare AI Example

```python
# ai-projects/healthcare-ai/src/healthcare_provider.py
from shared.chat_providers import BaseProvider
from typing import AsyncIterator
import json

class HealthcareAIProvider(BaseProvider):
    """Custom provider for medical query handling."""

    def __init__(self, model_id: str = "healthcare-gpt-4"):
        self.model_id = model_id
        self.system_prompt = """You are a medical education AI assistant.

        IMPORTANT CONSTRAINTS:
        - Always include disclaimer: "This is for educational purposes only"
        - For health conditions, recommend seeing a healthcare professional
        - Provide evidence-based information from reputable medical sources
        - Do not provide diagnosis or treatment recommendations
        - Flag potentially dangerous symptoms for immediate medical attention
        """

    async def complete(self, prompt: str) -> str:
        """Handle healthcare queries safely."""
        # Check for dangerous symptoms
        dangerous_symptoms = [
            "chest pain", "difficulty breathing", "severe bleeding",
            "unconscious", "suicidal", "severe allergic"
        ]

        for symptom in dangerous_symptoms:
            if symptom.lower() in prompt.lower():
                return f"⚠️ EMERGENCY: This requires immediate medical attention. Call 911 or go to nearest emergency room. {self.system_prompt}"

        # Process normal medical query
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": prompt}
        ]

        response = await self._call_llm(messages)
        return response

    async def stream(self, prompt: str) -> AsyncIterator[str]:
        """Stream healthcare query responses."""
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": prompt}
        ]

        async for chunk in self._stream_llm(messages):
            yield chunk

# Register provider
def register_healthcare_provider():
    from shared.chat_providers import PROVIDERS
    PROVIDERS["healthcare"] = HealthcareAIProvider()
```

### Financial AI Domain

```python
# ai-projects/finance-ai/src/finance_provider.py
import re
from decimal import Decimal

class FinanceAIProvider(BaseProvider):
    """Financial query handler with data validation."""

    def __init__(self):
        self.system_prompt = """You are a financial education AI.

        GUIDELINES:
        - Provide general financial education only
        - Include disclaimers about not being financial advice
        - Direct users to licensed financial advisors
        - Avoid recommending specific investments
        - Validate all calculations
        """

    def validate_financial_input(self, user_input: str) -> bool:
        """Validate financial queries for safety."""
        # Check for account numbers, SSN patterns
        if re.search(r'\b\d{9}\b', user_input):  # SSN pattern
            raise ValueError("Please don't share personal identification numbers")

        if re.search(r'credit.*card|account.*number', user_input, re.I):
            raise ValueError("Please don't share banking credentials")

        return True

    async def complete(self, prompt: str) -> str:
        """Handle financial query."""
        self.validate_financial_input(prompt)

        # Could integrate with:
        # - Financial data APIs (Alpha Vantage, IEX Cloud)
        # - Tax calculation engines
        # - Portfolio optimization algorithms

        response = await self._call_llm([
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": prompt}
        ])

        return response
```

### Education Domain

```python
# ai-projects/education-ai/src/education_provider.py
from enum import Enum

class EducationLevel(Enum):
    ELEMENTARY = "K-5"
    MIDDLE_SCHOOL = "6-8"
    HIGH_SCHOOL = "9-12"
    COLLEGE = "13-16"
    PROFESSIONAL = "17+"

class EducationAIProvider(BaseProvider):
    """Education-focused AI with adaptive learning."""

    def __init__(self, level: EducationLevel = EducationLevel.HIGH_SCHOOL):
        self.level = level
        self.system_prompts = {
            EducationLevel.ELEMENTARY: "Explain like I'm 8 years old",
            EducationLevel.MIDDLE_SCHOOL: "Use vocabulary for 11-14 year olds",
            EducationLevel.HIGH_SCHOOL: "Use standard high school vocabulary",
            EducationLevel.COLLEGE: "Use academic vocabulary",
            EducationLevel.PROFESSIONAL: "Use professional terminology"
        }

    async def complete(self, prompt: str) -> str:
        """Adaptive response based on education level."""
        system_prompt = self.system_prompts[self.level]

        response = await self._call_llm([
            {"role": "system", "content": f"{system_prompt}. Also include learning objectives and think about common misconceptions."},
            {"role": "user", "content": prompt}
        ])

        return response

    async def generate_quiz(self, topic: str, num_questions: int = 5):
        """Generate educational quiz."""
        prompt = f"Create a {num_questions}-question quiz about {topic} for {self.level.value} students"
        return await self.complete(prompt)

    async def get_learning_path(self, subject: str):
        """Suggest learning progression."""
        prompt = f"Create a learning path for {subject}. List prerequisites, core topics, and advanced topics."
        return await self.complete(prompt)
```

---

## Retrieval-Augmented Generation (RAG)

### Custom Retriever Plugin

```python
# ai-projects/healthcare-ai/src/medical_retriever.py
from typing import List
import numpy as np

class MedicalDocumentRetriever:
    """Retrieve relevant medical documents for queries."""

    def __init__(self, documents_source: str):
        self.documents_source = documents_source
        self.embeddings = None
        self.documents = None
        self.load_documents()

    def load_documents(self):
        """Load medical documents."""
        # Could load from:
        # - PubMed abstracts
        # - Medical textbooks
        # - Clinical guidelines
        # - Patient education materials

        import json
        with open(self.documents_source) as f:
            self.documents = json.load(f)

        # Generate embeddings
        self.embeddings = self._generate_embeddings(
            [doc["content"] for doc in self.documents]
        )

    def _generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for documents."""
        # Could use:
        # - OpenAI embeddings API
        # - Azure OpenAI embeddings
        # - Open-source models (sentence-transformers)

        # Example using shared memory
        from shared.chat_memory import ChatMemory
        memory = ChatMemory()
        embeddings = []
        for text in texts:
            embedding = memory.generate_embedding(text)
            embeddings.append(embedding)
        return np.array(embeddings)

    def retrieve_relevant_documents(self, query: str, top_k: int = 5) -> List[dict]:
        """Find most relevant medical documents."""
        from shared.chat_memory import ChatMemory
        memory = ChatMemory()

        # Get query embedding
        query_embedding = memory.generate_embedding(query)

        # Calculate similarity
        similarities = []
        for i, doc_embedding in enumerate(self.embeddings):
            similarity = np.dot(query_embedding, doc_embedding)
            similarities.append((i, similarity))

        # Sort by similarity and return top-k
        similarities.sort(key=lambda x: x[1], reverse=True)
        return [self.documents[i] for i, _ in similarities[:top_k]]

# Use in healthcare provider
class HealthcareProviderWithRAG(BaseProvider):
    def __init__(self):
        self.retriever = MedicalDocumentRetriever("data/medical_documents.json")

    async def complete(self, prompt: str) -> str:
        """Answer using retrieved documents."""
        # Retrieve relevant documents
        docs = self.retriever.retrieve_relevant_documents(prompt, top_k=3)

        # Augment prompt with context
        context = "\n\n".join([f"Reference: {doc['title']}\n{doc['content']}" for doc in docs])

        augmented_prompt = f"{prompt}\n\nRelevant medical references:\n{context}"

        # Generate response
        response = await self._call_llm([
            {"role": "system", "content": "Use the provided medical references to answer accurately."},
            {"role": "user", "content": augmented_prompt}
        ])

        return response
```

---

## Custom Tools & Integrations

### Tool Plugin System

```python
# ai-projects/finance-ai/src/tools/stock_price_tool.py
from typing import Dict

class StockPriceTool:
    """Tool for querying stock prices."""

    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = "https://api.example.com/stocks"

    async def get_price(self, ticker: str) -> Dict:
        """Get current stock price."""
        import aiohttp
        async with aiohttp.ClientSession() as session:
            url = f"{self.base_url}/{ticker}"
            async with session.get(url, headers={"Authorization": f"Bearer {self.api_key}"}) as resp:
                if resp.status == 200:
                    return await resp.json()
                else:
                    return {"error": f"Failed to fetch stock price for {ticker}"}

    def schema(self) -> Dict:
        """OpenAI function calling schema."""
        return {
            "name": "get_stock_price",
            "description": "Get current stock price for a given ticker symbol",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {
                        "type": "string",
                        "description": "Stock ticker symbol (e.g., AAPL, GOOGL)"
                    }
                },
                "required": ["ticker"]
            }
        }

# ai-projects/healthcare-ai/src/tools/symptom_checker_tool.py
class SymptomCheckerTool:
    """Tool for checking symptoms (educational only)."""

    def __init__(self, symptom_database: str):
        self.symptoms_db = self._load_symptom_database(symptom_database)

    async def check_symptoms(self, symptoms: List[str]) -> Dict:
        """Check symptoms against database."""
        results = []
        for symptom in symptoms:
            if symptom.lower() in self.symptoms_db:
                results.append({
                    "symptom": symptom,
                    "conditions": self.symptoms_db[symptom.lower()],
                    "urgency": self._determine_urgency(symptom)
                })
        return {"results": results}

    def _determine_urgency(self, symptom: str) -> str:
        """Emergency symptoms need immediate attention."""
        emergency_symptoms = [
            "chest pain", "difficulty breathing", "loss of consciousness",
            "severe bleeding", "severe allergic reaction"
        ]

        if symptom.lower() in emergency_symptoms:
            return "EMERGENCY - Call 911"
        return "Consult with healthcare provider"

    def schema(self) -> Dict:
        return {
            "name": "check_symptoms",
            "description": "Educational tool to check symptoms (not medical diagnosis)",
            "parameters": {
                "type": "object",
                "properties": {
                    "symptoms": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "List of symptoms to check"
                    }
                },
                "required": ["symptoms"]
            }
        }
```

---

## Deploying Custom Domains

### Package Structure

```
ai-projects/healthcare-ai/
├─ src/
│  ├─ healthcare_provider.py
│  ├─ medical_retriever.py
│  ├─ tools/
│  │  └─ symptom_checker_tool.py
│  └─ __init__.py
├─ data/
│  └─ medical_documents.json
├─ tests/
│  ├─ test_healthcare_provider.py
│  └─ test_medical_safety.py
├─ requirements.txt
└─ README.md
```

### Registration with Core

```python
# function_app.py
from ai_projects.healthcare_ai.src.healthcare_provider import HealthcareAIProvider

# Register custom provider
CUSTOM_PROVIDERS = {
    "healthcare": HealthcareAIProvider(),
    "finance": FinanceAIProvider(),
    "education": EducationAIProvider()
}

@app.route("/api/chat/domains", methods=["GET"])
async def list_domains(req: func.HttpRequest):
    """List available domain-specific providers."""
    return func.HttpResponse(
        json.dumps({
            "domains": list(CUSTOM_PROVIDERS.keys()),
            "default": "general"
        }),
        status_code=200
    )

@app.route("/api/chat/complete/<domain>", methods=["POST"])
async def domain_chat(req: func.HttpRequest, domain: str):
    """Chat with domain-specific provider."""
    if domain not in CUSTOM_PROVIDERS:
        return func.HttpResponse("Domain not found", status_code=404)

    provider = CUSTOM_PROVIDERS[domain]
    data = req.get_json()
    response = await provider.complete(data["message"])

    return func.HttpResponse(json.dumps({"response": response}))
```


## Best Practices for Custom Domains

### Safety & Compliance

- [ ] Add appropriate disclaimers
- [ ] Validate inputs for sensitive data
- [ ] Implement domain-specific guardrails
- [ ] Comply with industry regulations (HIPAA, SOC2, etc.)
- [ ] Audit sensitive operations
- [ ] Test edge cases and harmful inputs

### Performance

- [ ] Cache domain-specific data
- [ ] Preload reference documents
- [ ] Optimize retrieval queries
- [ ] Use connection pooling for external APIs
- [ ] Implement rate limiting
- [ ] Monitor domain-specific metrics

### Testing

- [ ] Unit tests for domain logic
- [ ] Safety tests for guardrails
- [ ] Integration tests with main platform
- [ ] Adversarial testing (jailbreak attempts)
- [ ] Compliance audits
- [ ] User acceptance testing

### Monitoring

- [ ] Track domain-specific metrics
- [ ] Log sensitive operations
- [ ] Alert on policy violations
- [ ] Monitor external API usage
- [ ] Track accuracy metrics
- [ ] User feedback collection
